# ML-Based Slope Stability Prediction in Lateritic Terrain

This notebook provides an interactive walkthrough of the analysis pipeline.

## 1. Load and Explore Original Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import RepeatedKFold, cross_val_score

# Load original field data
df_original = pd.read_csv('../data/raw/01b_original_data.csv')
print(f'Original dataset: {df_original.shape[0]} samples, {df_original.shape[1]} features')
df_original.describe()

## 2. Load Augmented Dataset

In [ ]:
# Load Monte Carlo augmented dataset
df = pd.read_csv('../data/augmented/01_augmented_dataset.csv')
print(f'Augmented dataset: {df.shape[0]} samples, {df.shape[1]} features')
print(f'\nFoS range: {df["FoS"].min():.3f} to {df["FoS"].max():.3f}')
print(f'FoS mean: {df["FoS"].mean():.3f} ± {df["FoS"].std():.3f}')
df.head(10)

## 3. Correlation Analysis

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Correlation with FoS
print('\nCorrelation with Factor of Safety:')
print(corr['FoS'].sort_values(ascending=False))

## 4. PCA Analysis

In [ ]:
# Prepare features
feature_cols = [c for c in df.columns if c not in ['FoS', 'Site']]
X = df[feature_cols].values
y = df['FoS'].values

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA()
pca.fit(X_scaled)

# Scree plot
fig, ax = plt.subplots(figsize=(10, 6))
components = range(1, len(pca.explained_variance_ratio_) + 1)
ax.bar(components, pca.explained_variance_ratio_ * 100, alpha=0.7, label='Individual')
ax.plot(components, np.cumsum(pca.explained_variance_ratio_) * 100, 'ro-', label='Cumulative')
ax.axhline(y=95, color='gray', linestyle='--', label='95% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('PCA Scree Plot')
ax.legend()
plt.tight_layout()
plt.show()

# Kaiser criterion
n_components = sum(pca.explained_variance_ > 1)
print(f'\nComponents with eigenvalue > 1: {n_components}')
print(f'Cumulative variance explained: {sum(pca.explained_variance_ratio_[:n_components])*100:.1f}%')

## 5. Feature Selection (RFE)

In [ ]:
# RFE with Random Forest
rf = RandomForestRegressor(n_estimators=500, random_state=42)
rfe = RFE(estimator=rf, n_features_to_select=5)
rfe.fit(X_scaled, y)

# Feature rankings
rankings = pd.DataFrame({
    'Feature': feature_cols,
    'Rank': rfe.ranking_,
    'Selected': rfe.support_
}).sort_values('Rank')

print('RFE Feature Rankings:')
print(rankings.to_string(index=False))
print(f'\nSelected features: {[feature_cols[i] for i in range(len(feature_cols)) if rfe.support_[i]]}')

## 6. Model Comparison

In [ ]:
# Use RFE Top-5 features
X_rfe = X_scaled[:, rfe.support_]

# Define models
models = {
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42),
    'SVM (RBF)': SVR(kernel='rbf'),
    'KNN (k=5)': KNeighborsRegressor(n_neighbors=5)
}

# Evaluate with repeated K-Fold
cv = RepeatedKFold(n_splits=10, n_repeats=5, random_state=42)
results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_rfe, y, cv=cv, scoring='r2')
    results[name] = scores
    print(f'{name:20s}: R² = {scores.mean():.4f} ± {scores.std():.4f}')

# Box plot comparison
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(results.values(), labels=results.keys())
ax.set_ylabel('Cross-Validated R²')
ax.set_title('Model Performance Comparison (RFE Top-5 Features)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 7. Decision Tree Interpretation

In [ ]:
from sklearn.tree import export_text

# Train a pruned decision tree for interpretability
dt = DecisionTreeRegressor(max_depth=3, random_state=42)
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if rfe.support_[i]]
dt.fit(X_rfe, y)

# Print tree rules
tree_rules = export_text(dt, feature_names=selected_features)
print('Decision Tree Rules (max_depth=3):')
print(tree_rules)